# Fig.3a 再現: transcriptomeベースの薬剤-疾患相関 boxplot

対象論文: Regulome-based characterization of drug activity across the human diseasome (npj Syst Biol Appl 2022, PMC9640590)

手順書: `docs/repro_regulome_fig3.md`

**ゴール**: 疾患ごとに「その疾患の承認薬」の薬剤-疾患コサイン相関スコアの分布をboxplot化。中央値昇順ソート。がん系=負相関、免疫系=正相関に分離するか確認する。

**方針**: L1000薬剤シグネチャと承認薬マッピングはSnowflake `RAW.LINCS` から取得（`data/raw/`に置かない）。疾患シグネチャ(CREEDS)のみ外部取得。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector as sc

# 作図規約は /figure-style skill を参照
pd.set_option('display.max_columns', 50)

# admin コネクション(RAW.LINCS/RAW.CREEDS)。config.tomlのprivate_key_pathはコネクタが直接読めないので明示する。
con = sc.connect(
    account='DUETMBM-LL33279', user='KOREEDA', role='ACCOUNTADMIN',
    warehouse='BIOINFORMATICS_XS', authenticator='SNOWFLAKE_JWT',
    private_key_file=os.path.expanduser('~/.ssh/snowflake_rsa_key.pem'),
)
cur = con.cursor()

def q(sql):
    df = cur.execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

def norm_gene(g):
    """遺伝子シンボル正規化: 大文字化 + '-'→'_'（L1000列は HLA_DMA 形式）。"""
    return str(g).upper().replace('-', '_')

## Step 1. 薬剤トランスクリプトームシグネチャ x_trans (LINCS L1000 Level5 MODZ)

- Snowflake `RAW.LINCS.L1000_LEVEL5_LANDMARK`（Level5 MODZ ワイド形式）から `PERT_TYPE='trt_cp'` を取得。
- メタ列: SAMPLE_ID, CELL, PERTNAME, PERTID, DOSE, TIMEPOINT, PERT_TYPE。残りが遺伝子FLOAT列（約959）。
- 各プロファイルで top5% / bottom5% 以外を 0 埋め。

> 接続は `admin` コネクション（`RAW.LINCS`）。CLIなら `snow sql -c admin`、コネクタなら `connection_name='admin'`。全473,647行・trt_cpは205,034行と大きいので、パイロットはCELL/PERTNAMEで絞る。

In [ ]:
# --- Step 1: 薬剤シグネチャ x_trans（化合物ごとに全trt_cp平均 → top/bottom 5%） ---
# 対象薬剤は承認薬マッピングビューに出てくるものだけ（Fig3aに必要な89薬剤）。
drug_list = q("SELECT DISTINCT drug_lc FROM RAW.CREEDS.VW_FIG3A_APPROVED_DRUG_MAP")['drug_lc'].tolist()
inlist = ",".join("'%s'" % d.replace("'", "''") for d in drug_list)
df_drug = q(f"""
  SELECT * FROM RAW.LINCS.L1000_LEVEL5_LANDMARK
  WHERE PERT_TYPE='trt_cp' AND LOWER(PERTNAME) IN ({inlist})
""")
META = ['sample_id','cell','pertname','pertid','dose','timepoint','pert_type']
gene_cols = [c for c in df_drug.columns if c not in META]
print(f"profiles={len(df_drug)} gene_cols={len(gene_cols)}")

def top_bottom_5pct_row(v):
    """top5%/bottom5%以外を0にするシグネチャ化。"""
    v = np.asarray(v, dtype=float)
    n = len(v); k = max(1, int(round(n * 0.05)))
    out = np.zeros_like(v)
    order = np.argsort(v)
    out[order[:k]] = v[order[:k]]      # bottom 5%
    out[order[-k:]] = v[order[-k:]]    # top 5%
    return out

# 化合物ごとに raw MODZ を平均 → top/bottom 5% 化
df_drug['pert_lc'] = df_drug['pertname'].str.lower()
drug_raw = df_drug.groupby('pert_lc')[gene_cols].mean()
drug_sig = pd.DataFrame(
    {idx: top_bottom_5pct_row(drug_raw.loc[idx].values) for idx in drug_raw.index},
    index=[norm_gene(g) for g in gene_cols]
).T   # drugs x genes(normalized), thresholded
drug_sig.shape

## Step 2. 疾患トランスクリプトームシグネチャ y_trans (CREEDS → Snowflake `RAW.CREEDS`)

- 取り込み済み（`docs/plan_creeds_to_snowflake.md` 参照）。全828シグネチャ、DOID付き695、ヒット202/396等がJSON原本と一致確認済み。
- 解析は `RAW.CREEDS.DISEASE_SIGNATURE_GENES`（signature_id, gene, value, direction）を使用。value符号=方向・絶対値=Characteristic Direction重み。
- ヒト＋DOID付きに絞ったビュー `RAW.CREEDS.VW_HUMAN_DISEASE_GENES` あり（ヒト555, 疾患DOID 178種）。
- 同一疾患(do_id)の複数シグネチャは解析側で平均化。

In [ ]:
# --- Step 2: 疾患シグネチャ y_trans（同一do_idの複数CREEDSシグネチャを平均、0-fill） ---
df_dis = q("""
  SELECT g.signature_id, g.gene, g.value, m.do_id
  FROM RAW.CREEDS.VW_HUMAN_DISEASE_GENES g
  JOIN RAW.CREEDS.DISEASE_SIGNATURES_META m USING (signature_id)
""")
df_dis['gene'] = df_dis['gene'].map(norm_gene)

# シグネチャベクトルの平均 = sum(value)/n_sig（欠損遺伝子は0扱い）
n_sig = df_dis.groupby('do_id')['signature_id'].nunique()
dis_sum = df_dis.groupby(['do_id', 'gene'])['value'].sum().reset_index()
dis_sum['avg'] = dis_sum['value'] / dis_sum['do_id'].map(n_sig)
dis_sig = dis_sum.pivot(index='do_id', columns='gene', values='avg').fillna(0.0)
print(f"do_ids={dis_sig.shape[0]} genes={dis_sig.shape[1]}")
dis_sig.iloc[:3, :5]

## Step 3. 薬剤-疾患コサイン相関

共通遺伝子（論文≈848/14,639）に揃えて cosine 類似度。

In [ ]:
# --- Step 3: 共通遺伝子でコサイン相関 ---
common = sorted(set(drug_sig.columns) & set(dis_sig.columns))
print(f"common genes = {len(common)}")   # 論文≈848
D = drug_sig[common]
Y = dis_sig[common]

def cosine(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return np.nan
    return float(np.dot(u, v) / (nu * nv))

# 承認薬マッピング上の各(do_id, pertname)ペアで cosine
rows = []
for _, r in df_map.iterrows():
    do, dz, pl = r['do_id'], r['disease_name'], r['drug_lc']
    if pl in D.index and do in Y.index:
        rows.append((do, dz, pl, cosine(D.loc[pl].values, Y.loc[do].values)))
res = pd.DataFrame(rows, columns=['do_id', 'disease', 'drug', 'cos']).dropna(subset=['cos'])
print(f"pairs_with_cos={len(res)} diseases={res['do_id'].nunique()}")

# 保存
out_csv = '../../results/tables/fig3a_repro_correlations.csv'
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
res.to_csv(out_csv, index=False)
res.head()

## Step 4. 疾患↔承認薬マッピング（構築済みビュー `RAW.CREEDS.VW_FIG3A_APPROVED_DRUG_MAP`）

クロスワーク構築・検証済み（2026-07-14）。**41疾患 / 89薬剤 / 130ペア**。

- 定義: CREEDS(ヒト+DOID)`disease_name` × ChEMBL承認適応(MAX_PHASE=4, MeSH/EFO) を正規化文字列一致で突合 → さらに L1000 `PERTNAME` に存在する薬剤に限定。
- カラム: `do_id, disease_name, pertname, drug_lc, match_source`。
- がん系（乳がん・前立腺がん・腎細胞がん・多発性骨髄腫）と免疫系（関節リウマチ・乾癬・喘息）を両方含む → がん負/免疫正パターンの検証が可能。

In [ ]:
# --- Step 4: 承認薬マッピング（構築済みビュー） ---
df_map = q("SELECT do_id, disease_name, pertname, drug_lc FROM RAW.CREEDS.VW_FIG3A_APPROVED_DRUG_MAP")
drugs_lc = sorted(df_map['drug_lc'].unique())
print(f"diseases={df_map['do_id'].nunique()} drugs={len(drugs_lc)} pairs={len(df_map)}")
df_map.head()

## Step 5. 作図 (Fig.3a)

中央値昇順ソートのboxplot。がん系(下)/免疫系(上)の分離を確認。

In [ ]:
# --- Step 5: Fig3a boxplot（中央値昇順、がん=赤/免疫=青で色分け） ---
CANCER = ['cancer','carcinoma','leukemia','lymphoma','myeloma','melanoma','glioma',
          'sarcoma','astrocytoma','adenoma','tumor','blastoma']
IMMUNE = ['arthritis','psoriasis','asthma','crohn','colitis','lupus','dermatitis',
          'inflammatory','multiple sclerosis']
def cat(name):
    n = name.lower()
    if any(k in n for k in CANCER): return 'cancer'
    if any(k in n for k in IMMUNE): return 'immune'
    return 'other'

med = res.groupby('disease')['cos'].median().sort_values()
order = med.index.tolist()
data = [res.loc[res['disease'] == d, 'cos'].values for d in order]
catcol = {'cancer': '#c0392b', 'immune': '#2471a3', 'other': '#7f8c8d'}
colors = [catcol[cat(d)] for d in order]

fig, ax = plt.subplots(figsize=(15, 5))
bp = ax.boxplot(data, showfliers=True, widths=0.6, patch_artist=True)
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col); patch.set_alpha(0.55)
ax.axhline(0, color='grey', lw=0.6, ls='--')
ax.set_xticks(range(1, len(order) + 1))
ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_ylabel('Cosine correlation (transcriptome)')
ax.set_title('Fig.3a reproduction (red=cancer, blue=immune)')

out_fig = '../../results/figures/fig3a_repro.png'
os.makedirs(os.path.dirname(out_fig), exist_ok=True)
fig.tight_layout(); fig.savefig(out_fig, dpi=200, bbox_inches='tight')
print('カテゴリ別中央値:'); print(res.assign(cat=res['disease'].map(cat)).groupby('cat')['cos'].median())
plt.show()